# Parse ENTOSE National Net Generation Capacities

Notebook to parse national Net Generation Capacities provided by ENTSOE:
https://www.entsoe.eu/data/power-stats/net-gen-capacity/

hydro capacities added from hydro_ngc but replaced with JRC data in create gdx
JSA: 27/10/20

Notebook updated 23.07.25
Need to check if the capacities are still used somewhere...



## Packages and options

In [1]:
import pandas as pd
import numpy as np

c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [5]:
fn_in = "../source_data/NGC.xlsx"
fn_in_fixed_hydro = "../source_data/hydro_ngc.xlsx"
dir_out = "../parsed_data/"

Dictionary to rename technologies names by ENTSOE to own names. Note that we only rename the lowest technology level and drop all aggregates:

In [6]:
col_rename = {'Geothermal': 'geothermal',
        'Non identified (other not listed)': 'not_identified',
        'Non-renewable Waste': 'waste',
        'Nuclear': 'nuclear',
        'Of which Biogas': 'biogas',
        'Of which Biomass': 'biomass',
        'Of which Fossil Brown coal/Lignite': 'lignite',
        'Of which Fossil Coal-derived gas': 'coal_gas',
        'Of which Fossil Gas': 'gas',
        'Of which Fossil Hard coal': 'coal',
        'Of which Fossil Oil': 'oil',
        'Of which Fossil Oil shale': 'oil_shale',
        'Of which Fossil Peat': 'peat',
        'Of which Hydro Marine (tidal/wave)': 'tidal',
        'Of which Hydro Pure storage': 'hydro_storage',
        'Of which Hydro Run-of-river and pondage': 'hydro_ror',
        'Of which Hydro mixed pumped storage (non renewable part)': 'hydro_pump_mixed',
        'Of which Hydro mixed pumped storage (renewable part)': 'hydro_pump_mixed_renewable',
        'Of which Mixed fuels': 'mixed',
        'Of which Other fossil fuels': 'other_fossil',
        'Of which Solar PV': 'solar_pv',
        'Of which Solar Thermal': 'solar_thermal',
        'Of which Wind offshore': 'wind_offshore',
        'Of which Wind onshore': 'wind_onshore',
        'Of which hydro pure pumped storage': 'hydro_pump_pure',
        'Other non-renewable': 'other',
        'Other renewable (not listed)': 'other_renewable',
        'Renewable Waste': 'waste_renewable',
        'year': 'year'}

# aggregate technologies to model level
dict_agg_tech = {'geothermal': 'Other',
                'not_identified': 'Other',
                'waste': 'Other',
                'nuclear': 'Nuclear',
                'biogas': 'Biomass',
                'biomass': 'Biomass',
                'lignite': 'Lignite',
                'coal_gas': 'Other',
                'gas': 'Gas',
                'coal': 'HardCoal',
                'oil': 'Oil',
                'oil_shale': 'Oil',
                'peat': 'Other',
                'tidal': 'Other',
                'hydro_storage': 'Reservoir',
                'hydro_ror': 'RunOfRiver',
                'hydro_pump_mixed': 'Pump',
                'hydro_pump_mixed_renewable': 'Pump',
                'mixed': 'Other',
                'other_fossil': 'Other',
                'solar_pv': 'Solar',
                'solar_thermal': 'Solar',
                'wind_offshore': 'WindOffshore',
                'wind_onshore': 'WindOnshore',
                'hydro_pump_pure': 'Pump',
                'other': 'Other',
                'other_renewable': 'Other',
                'waste_renewable': 'Other'
                }

## Parse Excel File

In [7]:
df_ = pd.read_excel(fn_in, skiprows=9, header=[0,1], na_values=["Not Expected", "Not Available"])
df_ = df_.set_index(df_.iloc[:,0]) #added by Jonas to make this work with Python 3

# get country order from header and 
countries = list(df_.columns.levels[0])
df_.columns = df_.columns.droplevel(0)
first_year = int(countries[0])
countries = countries[1:]

# get only columns with MW values
df_ = df_[["Value in MW"]].copy()

# assign country headers
df_.columns = countries

# delete emtpy rows
df_ = df_[df_.index.notnull()].copy()

# get rows with comments marking the en of a year
idx_comments = df_.index[df_.index == "Comments"]
idx_comments = [i for i,v in enumerate(df_.index) if v == "Comments"]

# split frames 
first = 0
year = first_year
lst_df = []
for i in idx_comments:
    df_1 = df_.iloc[first:i].T.copy()
    df_1["year"] = year
    
    # if there is a column named same as the year we looking at, drop it
    df_1 = df_1[[c for c in df_1.columns if c != year]]
    
    lst_df.append(df_1)
    first = i + 1
    year += 1
df_in = pd.concat(lst_df, sort=True)
df_in.info()

<class 'pandas.core.frame.DataFrame'>
Index: 144 entries, AL to TR
Data columns (total 40 columns):
 #   Column                                                    Non-Null Count  Dtype 
---  ------                                                    --------------  ----- 
 0   Bio                                                       144 non-null    object
 1   Fossil fuels                                              144 non-null    object
 2   Geothermal                                                18 non-null     object
 3   Non identified (other not listed)                         9 non-null      object
 4   Non-Renewable                                             144 non-null    object
 5   Non-renewable Waste                                       39 non-null     object
 6   Non-renwable hydro                                        144 non-null    object
 7   Nuclear                                                   60 non-null     object
 8   Of which Biogas                    

Rename columns, drop aggregated columns:

In [8]:
df_cap = df_in.rename(columns=col_rename)[list(col_rename.values())].replace({None: np.nan})
df_cap.index.name = "country"
df_cap = df_cap.reset_index().set_index(["year", "country",])
df_cap = df_cap.applymap(lambda x: float(x))
df_cap.sum(1)

C:\Users\jonas\AppData\Local\Temp\ipykernel_33104\2957368150.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cap = df_in.rename(columns=col_rename)[list(col_rename.values())].replace({None: np.nan})
C:\Users\jonas\AppData\Local\Temp\ipykernel_33104\2957368150.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_cap = df_cap.applymap(lambda x: float(x))


year  country
2014  AL             0.00
      AT         24222.45
      BA          3638.00
      BE         20114.00
      BG         13563.00
                   ...   
2017  RS          8494.19
      SE         39037.00
      SI          3816.22
      SK          7702.00
      TR         85200.10
Length: 144, dtype: float64

In [9]:
df_out = df_cap.stack().reset_index()
df_out.columns = ["year", "country", "technology", "MW"]
df_out.head(1)

,year,country,technology,MW
0,2014,AT,geothermal,1.0


## Aggregate technologies

In [10]:
df_agg = df_out.copy()
df_agg["tech_new"] = df_agg.technology.map(dict_agg_tech)
df_agg = df_agg.groupby(["year", "country", "tech_new"], as_index=False).MW.sum()
df_agg.columns = ["year", "country", "technology", "capacity"]
df_agg = df_agg[df_agg.year.isin(range(2015, 2019))]
df_agg.info()

<class 'pandas.core.frame.DataFrame'>
Index: 876 entries, 286 to 1161
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   year        876 non-null    int64  
 1   country     876 non-null    object 
 2   technology  876 non-null    object 
 3   capacity    876 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 34.2+ KB


# Load Hydro capacities from different source

Substitute capacity values for model countries with data from entso-e transparency platform and national statistics data.
Skip this section for retaining previous capacity values.

In [11]:
countries = ["AT", "BE", "CH", "CZ", "DE", "DK", "ES", "FI", "FR", "GB", "IE", "IT", "LU", "NL", "NO", "PL", "PT", "SE"]

In [12]:
df_fixed_hydro = df_agg.copy()
mask = (((df_fixed_hydro.technology == "RunOfRiver") | (df_fixed_hydro.technology == "Reservoir") 
                                 |(df_fixed_hydro.technology == "Pump")) & (df_fixed_hydro.year == 2017) 
                                 & (df_fixed_hydro["country"].isin(countries)))
df_fixed_hydro = df_fixed_hydro.drop(df_fixed_hydro.loc[mask].index)

In [13]:
df_fixed_hydro.head()

,year,country,technology,capacity
286,2015,AT,Biomass,596.0
287,2015,AT,Gas,4820.0
288,2015,AT,HardCoal,1171.0
289,2015,AT,Oil,174.0
290,2015,AT,Other,1008.0


In [20]:
df_new_hydro_in = pd.read_excel(fn_in_fixed_hydro)

In [21]:
#df_fixed_hydro = df_fixed_hydro.append(df_new_hydro_in) append is deprecated in pandas
#use pd.concat instead
df_fixed_hydro = pd.concat([df_fixed_hydro, df_new_hydro_in], ignore_index=True)
df_fixed_hydro = df_fixed_hydro.reset_index().drop("index", axis = 1)
df_fixed_hydro = df_fixed_hydro.sort_values(by=["year", "country"])

In [22]:
df_agg = df_fixed_hydro.copy()

## Export data

In [23]:
df_agg.to_csv(dir_out + "capacities.csv", encoding="utf-8", index=False)

In [24]:
df_new_hydro_in.country.unique()

array(['AT', 'CH', 'CZ', 'DE', 'ES', 'FR', 'GB', 'IT', 'LU', 'NO', 'PL',
       'PT', 'SE', 'BE', 'DK', 'FI', 'IE', 'NL'], dtype=object)

In [25]:
df_agg.country.unique()

array(['AT', 'BA', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES',
       'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LT', 'LU',
       'LV', 'ME', 'MK', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI',
       'SK', 'TR', 'AL'], dtype=object)